# Getaround : analyse des retards et des prix

Ce notebook répond à deux questions :
1. **Les retards** : faut-il imposer un délai minimum entre deux locations, et de combien de minutes ?
2. **Le prix** : à quoi ressemblent les prix de location ? (le modèle de prix est dans `train.py`)

Les données sont dans le dossier `data/` :
- `get_around_delay_analysis (1).xlsx` : les locations et leurs retards
- `get_around_pricing_project (1).csv` : les voitures et leur prix par jour

In [9]:
import pandas as pd
import plotly.express as px

# Chargement des locations (le fichier est dans le dossier data, au-dessus de notebooks)
df = pd.read_excel("../data/get_around_delay_analysis (1).xlsx")

print("Nombre de lignes et de colonnes :", df.shape)
df.head()

Nombre de lignes et de colonnes : (21310, 7)


,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [10]:
# Les colonnes, leur type, et combien de cases sont remplies
df.info()

# Les deux façons de faire le check-in : mobile, ou connect (Getaround Connect)
print(df["checkin_type"].value_counts())

# Combien de cases vides par colonne
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 21310 entries, 0 to 21309
Data columns (total 7 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   rental_id                                   21310 non-null  int64  
 1   car_id                                      21310 non-null  int64  
 2   checkin_type                                21310 non-null  str    
 3   state                                       21310 non-null  str    
 4   delay_at_checkout_in_minutes                16346 non-null  float64
 5   previous_ended_rental_id                    1841 non-null   float64
 6   time_delta_with_previous_rental_in_minutes  1841 non-null   float64
dtypes: float64(3), int64(2), str(2)
memory usage: 1.4 MB
checkin_type
mobile     17003
connect     4307
Name: count, dtype: int64


rental_id                                         0
car_id                                            0
checkin_type                                      0
state                                             0
delay_at_checkout_in_minutes                   4964
previous_ended_rental_id                      19469
time_delta_with_previous_rental_in_minutes    19469
dtype: int64

In [11]:
# Combien de locations terminées (ended) et annulées (canceled)
print(df["state"].value_counts())
print()

# Pour les retards, on garde seulement les locations où le retard est connu
retards = df["delay_at_checkout_in_minutes"].dropna()

print("Locations avec un retard connu :", len(retards))
print("Part en retard (plus de 0 minute) :", round((retards > 0).mean() * 100, 1), "%")
print("Part en retard de plus de 60 minutes :", round((retards > 60).mean() * 100, 1), "%")
print("Retard médian :", retards.median(), "minutes")

state
ended       18045
canceled     3265
Name: count, dtype: int64

Locations avec un retard connu : 16346
Part en retard (plus de 0 minute) : 57.5 %
Part en retard de plus de 60 minutes : 26.8 %
Retard médian : 9.0 minutes


In [12]:
# Seules les locations qui ont une location juste avant peuvent être touchées par un délai minimum
df_suite = df.dropna(subset=["time_delta_with_previous_rental_in_minutes"]).copy()
print("Locations avec une location juste avant :", len(df_suite))

# Pour chaque délai minimum, on compte les locations dont le délai avec la précédente est plus court
resultats = []
for seuil in [0, 30, 60, 90, 120]:
    impactees = (df_suite["time_delta_with_previous_rental_in_minutes"] < seuil).sum()
    part = round(impactees / len(df_suite) * 100, 1)
    resultats.append({"Délai minimum (min)": seuil, "Locations impactées": impactees, "Part (%)": part})

pd.DataFrame(resultats)

Locations avec une location juste avant : 1841


,Délai minimum (min),Locations impactées,Part (%)
0,0,0,0.0
1,30,279,15.2
2,60,401,21.8
3,90,584,31.7
4,120,666,36.2


In [13]:
# On retrouve, pour chaque location, le retard de la location qui la précède
retard_precedente = df[["rental_id", "delay_at_checkout_in_minutes"]].rename(
    columns={"rental_id": "previous_ended_rental_id",
             "delay_at_checkout_in_minutes": "retard_precedente"})
df_suite = df_suite.merge(retard_precedente, on="previous_ended_rental_id", how="left")

# Cas à problème : la précédente rend la voiture plus tard que le délai prévu entre les deux
df_suite["probleme"] = df_suite["retard_precedente"] > df_suite["time_delta_with_previous_rental_in_minutes"]

print("Retard de la précédente inconnu :", df_suite["retard_precedente"].isna().sum())
print("Cas à problème :", df_suite["probleme"].sum())

Retard de la précédente inconnu : 112
Cas à problème : 218


In [14]:
# Pour chaque délai minimum : combien de locations on bloque, et combien de vrais problèmes on évite
total_problemes = df_suite["probleme"].sum()

resultats = []
for seuil in [30, 60, 90, 120]:
    bloquees = df_suite["time_delta_with_previous_rental_in_minutes"] < seuil
    nb_bloquees = bloquees.sum()
    nb_resolus = (bloquees & df_suite["probleme"]).sum()
    resultats.append({
        "Délai minimum (min)": seuil,
        "Locations bloquées": nb_bloquees,
        "Problèmes évités": nb_resolus,
        "Part des problèmes évités (%)": round(nb_resolus / total_problemes * 100, 1),
        "Bloquées sans problème": nb_bloquees - nb_resolus
    })

pd.DataFrame(resultats)

,Délai minimum (min),Locations bloquées,Problèmes évités,Part des problèmes évités (%),Bloquées sans problème
0,30,279,116,53.2,163
1,60,401,146,67.0,255
2,90,584,172,78.9,412
3,120,666,180,82.6,486


In [15]:
# On compare deux façons d'appliquer la règle : à toutes les voitures, ou seulement à celles en check-in Connect
lignes = []
groupes = [("Toutes les voitures", df_suite),
           ("Connect seulement", df_suite[df_suite["checkin_type"] == "connect"])]

for nom, sous in groupes:
    total_p = sous["probleme"].sum()
    for seuil in [30, 60, 90]:
        bloquees = sous["time_delta_with_previous_rental_in_minutes"] < seuil
        resolus = (bloquees & sous["probleme"]).sum()
        lignes.append({
            "Voitures": nom,
            "Délai (min)": seuil,
            "Locations bloquées": bloquees.sum(),
            "Problèmes évités": resolus,
            "Part évitée (%)": round(resolus / total_p * 100, 1)
        })

pd.DataFrame(lignes)

,Voitures,Délai (min),Locations bloquées,Problèmes évités,Part évitée (%)
0,Toutes les voitures,30,279,116,53.2
1,Toutes les voitures,60,401,146,67.0
2,Toutes les voitures,90,584,172,78.9
3,Connect seulement,30,131,40,58.0
4,Connect seulement,60,181,48,69.6
5,Connect seulement,90,260,56,81.2


In [16]:
# Données de prix : une ligne = une voiture
df_prix = pd.read_csv("../data/get_around_pricing_project (1).csv")
df_prix = df_prix.drop("Unnamed: 0", axis=1, errors="ignore")

print("Nombre de voitures :", len(df_prix))
print()

# Résumé du prix par jour : moyenne, minimum, maximum...
print(df_prix["rental_price_per_day"].describe().round(1))
print()

# Prix moyen selon le carburant
print(df_prix.groupby("fuel")["rental_price_per_day"].mean().round(1))
print()

# Lien entre prix, kilométrage et puissance (de -1 à 1)
df_prix[["rental_price_per_day", "mileage", "engine_power"]].corr().round(2)

Nombre de voitures : 4843

count    4843.0
mean      121.2
std        33.6
min        10.0
25%       104.0
50%       119.0
75%       136.0
max       422.0
Name: rental_price_per_day, dtype: float64

fuel
diesel           121.5
electro          145.0
hybrid_petrol    184.8
petrol           110.1
Name: rental_price_per_day, dtype: float64



,rental_price_per_day,mileage,engine_power
rental_price_per_day,1.00,-0.45,0.63
mileage,-0.45,1.00,-0.05
engine_power,0.63,-0.05,1.00


# Conclusion

## Les retards
- **57,5 %** des retours sont en retard, mais le retard médian est de seulement **9 minutes**. **26,8 %** des retours ont plus d'une heure de retard.
- Seules **1 841 locations** (environ 9 % des 21 310) ont une location juste avant : c'est sur elles qu'un délai minimum a un effet.
- Dans **218 cas**, la voiture précédente est rendue après le début prévu de la suivante : le conducteur suivant attend.

## Quel délai minimum ?

| Délai | Locations bloquées | Problèmes évités |
|---|---|---|
| 30 min | 279 (15,2 %) | 116 (53 %) |
| 60 min | 401 (21,8 %) | 146 (67 %) |
| 90 min | 584 (31,7 %) | 172 (79 %) |
| 120 min | 666 (36,2 %) | 180 (83 %) |

Plus le délai augmente, moins il rapporte : de 60 à 90 minutes, on évite seulement 26 problèmes de plus, mais on bloque 183 locations de plus. **Je recommande un délai de 30 à 60 minutes.**

Appliquer la règle **seulement aux voitures Connect** bloque moins de locations (131 à 30 minutes), mais évite aussi moins de problèmes (40 contre 116 pour toutes les voitures).

## Le prix
- 4 843 voitures, prix moyen **121 par jour**.
- Le modèle XGBoost de `train.py` a un **R² de 0,748** : il explique environ 75 % des différences de prix.

## Mes limites
- Pour **112 locations**, le retard de la précédente est inconnu : je ne les compte pas comme problèmes.
- Je suppose qu'une location bloquée n'a pas lieu. En réalité, le client pourrait choisir un autre horaire.
- Je compte des locations, pas de l'argent : je ne mesure pas le chiffre d'affaires perdu.
- Les prix minimum (10) et maximum (422) sont des valeurs extrêmes que je n'ai pas retirées.